# Knowledge Retrieval (RAG)

Retrieval-Augmented Generation (RAG) grounds LLM responses in external, up-to-date knowledge. The pattern: retrieve relevant document chunks → augment the prompt → generate a grounded answer.

## Implementation with Flyte v2

This notebook reimplements the LangChain + LangGraph `StateGraph` RAG pipeline from Chapter 14 using **Flyte v2 primitives only**. Weaviate is replaced with ChromaDB (in-memory, zero infrastructure) so the notebook runs end-to-end without external services.

#### LangChain / LangGraph vs Flyte v2 — Key Differences

| Aspect | LangChain / LangGraph | Flyte v2 |
|--------|----------------------|----------|
| **Graph definition** | `StateGraph` with `add_node` / `add_edge` | Plain Python tasks composed sequentially |
| **State type** | `TypedDict` (`RAGGraphState`) | Typed dataclasses — serializable, inspectable in UI |
| **Vector store** | Weaviate (external service required) | ChromaDB in-memory (zero infrastructure) |
| **LLM client** | `ChatOpenAI` / `langchain_openai` | Direct `AsyncOpenAI` client |
| **Caching** | None | `cache="auto"` — skips re-embedding unchanged documents |
| **Execution** | In-process only | Local or remote (containers) |
| **Secrets** | `.env` / `os.environ` | `flyte.Secret` injected by cluster |

### 1. Install dependencies

In [ ]:
!uv pip install 'flyte[tui]' openai chromadb

### Start the devbox

If you haven't already, install the flyte package with the command above, then launch the local cluster:

In [ ]:
!flyte start devbox

### 2. Store your API key

In [ ]:
!flyte create secret OPENAI_API_KEY --value sk-proj-...

### 3. Import dependencies and configure the TaskEnvironment

In [ ]:
from __future__ import annotations

import os
from dataclasses import dataclass, field
from datetime import timedelta

import flyte

flyte.init_from_config()

_image = (
    flyte.Image.from_debian_base(name="rag-agent", python_version=(3, 12))
    .with_pip_packages("openai>=1.0.0", "chromadb>=0.5.0")
)

rag_env = flyte.TaskEnvironment(
    name="rag_pipeline",
    image=_image,
    resources=flyte.Resources(cpu="1", memory="2Gi"),
    secrets=[
        flyte.Secret(key="OPENAI_API_KEY", as_env_var="OPENAI_API_KEY"),
    ],
)

### 4. Define data models

The LangGraph `StateGraph` used a `TypedDict` (`RAGGraphState`) to pass state between nodes. In Flyte v2, typed dataclasses serve the same purpose but are additionally:
- Serialized to object storage between tasks (no in-memory dependency)
- Visible as structured outputs in the UI
- Passable to any downstream task or workflow

In [ ]:
@dataclass
class DocumentChunk:
    """A single chunk of text from the source document."""
    text: str
    source: str = ""
    chunk_id: int = 0


@dataclass
class RetrievedContext:
    """Retrieved document chunks for a given question."""
    question: str
    chunks: list[DocumentChunk] = field(default_factory=list)

    def format_context(self) -> str:
        return "\n\n".join(chunk.text for chunk in self.chunks)


@dataclass
class RAGResult:
    """Final output of the RAG pipeline."""
    question: str
    answer: str
    retrieved_chunks: int
    sources: list[str] = field(default_factory=list)

### 5. Define the RAG pipeline tasks

The LangGraph `StateGraph` had two nodes: `retrieve_documents_node` and `generate_response_node`. In Flyte v2, these become two tasks that compose naturally via Python.

**Why no `StateGraph`?** LangGraph's graph is a way to express sequential or conditional pipelines with typed state. Flyte tasks already provide this: each task receives typed inputs and produces typed outputs. For a linear pipeline (retrieve → generate), plain Python composition is simpler and more readable.

**Caching:** The `retrieve_task` uses `cache="auto"` — if the same document corpus and question are re-run (e.g., repeated evaluation), Flyte skips re-embedding and returns the cached context instantly.

In [ ]:
def _chunk_text(text: str, chunk_size: int = 500, overlap: int = 50) -> list[str]:
    """Split text into overlapping chunks."""
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        # Find a sentence boundary near the end
        boundary = text.rfind(".", start, end)
        if boundary > start + chunk_size // 2:
            end = boundary + 1
        chunks.append(text[start:end].strip())
        start = end - overlap
    return [c for c in chunks if c]


@rag_env.task(cache="auto")
async def retrieve_task(
    document_text: str,
    question: str,
    top_k: int = 3,
) -> RetrievedContext:
    """
    Retrieve relevant chunks for the question.

    Replaces LangGraph's retrieve_documents_node.
    Uses ChromaDB in-memory — no external vector DB required.
    """
    import chromadb
    from openai import AsyncOpenAI

    client = AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"])

    # Chunk the document
    raw_chunks = _chunk_text(document_text)
    chunks = [
        DocumentChunk(text=c, source="document", chunk_id=i)
        for i, c in enumerate(raw_chunks)
    ]

    # Embed all chunks
    texts = [c.text for c in chunks]
    embed_response = await client.embeddings.create(
        model="text-embedding-3-small",
        input=texts,
    )
    chunk_embeddings = [e.embedding for e in embed_response.data]

    # Embed the query
    query_response = await client.embeddings.create(
        model="text-embedding-3-small",
        input=[question],
    )
    query_embedding = query_response.data[0].embedding

    # Store in ChromaDB in-memory and query
    chroma_client = chromadb.Client()
    collection = chroma_client.create_collection("rag_docs")
    collection.add(
        embeddings=chunk_embeddings,
        documents=texts,
        ids=[str(c.chunk_id) for c in chunks],
    )
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=min(top_k, len(chunks)),
    )

    retrieved = [
        DocumentChunk(text=doc, source="document", chunk_id=int(cid))
        for doc, cid in zip(results["documents"][0], results["ids"][0])
    ]

    return RetrievedContext(question=question, chunks=retrieved)


@rag_env.task(cache=flyte.Cache(behavior="disable"))
async def generate_task(context: RetrievedContext) -> RAGResult:
    """
    Generate an answer from retrieved context.

    Replaces LangGraph's generate_response_node.
    """
    from openai import AsyncOpenAI

    client = AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"])

    template = (
        "You are an assistant for question-answering tasks. "
        "Use the following retrieved context to answer the question. "
        "If you don't know the answer, say so. Keep the answer concise (3 sentences max).\n\n"
        f"Context:\n{context.format_context()}\n\n"
        f"Question: {context.question}\n\nAnswer:"
    )

    response = await client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": template}],
        temperature=0,
    )
    answer = (response.choices[0].message.content or "").strip()

    return RAGResult(
        question=context.question,
        answer=answer,
        retrieved_chunks=len(context.chunks),
        sources=list({c.source for c in context.chunks}),
    )

### 6. Orchestrate the pipeline

The LangGraph pipeline used `app.stream(inputs)` to execute the graph. In Flyte v2, we compose the two tasks directly inside a parent task — simpler and no graph definition boilerplate.

The parent task can run both tasks locally (via `asyncio.gather` for independent calls) or schedule them as separate container executions remotely.

In [ ]:
@rag_env.task(cache=flyte.Cache(behavior="disable"))
async def rag_pipeline(document_text: str, question: str, top_k: int = 3) -> RAGResult:
    """End-to-end RAG: retrieve relevant chunks, then generate a grounded answer."""
    context = await retrieve_task(document_text=document_text, question=question, top_k=top_k)
    return await generate_task(context=context)

### 7. Run locally

In [ ]:
DOCUMENT = """
Flyte is a cloud-native workflow orchestration platform designed for machine learning and data pipelines.
It was originally developed at Lyft and open-sourced in 2021. Flyte provides a type-safe, reproducible,
and scalable way to define and execute workflows using Python. Tasks in Flyte are containerized functions
that run on Kubernetes. Each task can be assigned specific compute resources like CPU, memory, and GPU.
Flyte supports caching of task outputs, which means if a task has already run with the same inputs, it
returns the cached result instead of recomputing. The Flyte v2 SDK uses a TaskEnvironment to group tasks
that share the same container image and resource configuration. Flyte also supports reusable containers
through ReusePolicy, which keeps warm containers around to reduce cold-start latency. Secrets are managed
securely through the flyte.Secret API, which injects credentials as environment variables at task execution
time without storing them in code. The Flyte UI provides real-time visibility into workflow execution,
including task inputs, outputs, logs, and custom HTML reports.
"""

QUESTIONS = [
    "What is Flyte and where was it originally developed?",
    "How does Flyte handle secrets?",
    "What is a TaskEnvironment in Flyte v2?",
]

for question in QUESTIONS:
    run = flyte.run(rag_pipeline, document_text=DOCUMENT, question=question, top_k=3)
    run.wait()
    result: RAGResult = run.outputs()[0]
    print(f"Q: {result.question}")
    print(f"A: {result.answer}")
    print(f"   (retrieved {result.retrieved_chunks} chunks)")
    print()

### Running remotely

With `cache="auto"` on `retrieve_task`, repeated queries against the same document corpus skip re-embedding — significantly reducing latency and API cost on subsequent runs.

In [ ]:
run = flyte.run(
    rag_pipeline,
    document_text=DOCUMENT,
    question="How does caching work in Flyte?",
    top_k=3,
)
run.wait()
result = run.outputs()[0]
print(f"Q: {result.question}")
print(f"A: {result.answer}")